In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install menpo
!git clone https://github.com/im-xiaoming/HuyenLaoNhao.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for menpo: filename=menpo-0.11.1-py3-none-any.whl size=1611934 sha256=ece4b9dcb4e48c60283d814308cec96f1bae999bf5b03936b0c27a5b2a2176c2
  Stored in directory: /root/.cache/pip/wheels/b7/98/ad/3e7933944c49e29b1f517f0998376cc8cc4d0f53a7c6537fb6
Successfully built menpo
Cloning into 'HuyenLaoNhao'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 140 (delta 71), reused 106 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 314.09 KiB | 34.90 MiB/s, done.
Resolving deltas: 100% (71/71), done.


In [ ]:
!cp -r /content/drive/MyDrive/Ying/data.zip /content/
!unzip /content/data.zip -d /content/
!cp -r /content/drive/MyDrive/Ying/mm.dat /content
!cp -r /content/drive/MyDrive/Ying/mm.dat.conf /content

Archive:  /content/data.zip
   creating: /content/data/
  inflating: /content/data/cfp_fp.dat  
  inflating: /content/data/lfw.dat.conf  
  inflating: /content/data/agedb_30_list.npy  
  inflating: /content/data/lfw_list.npy  
  inflating: /content/data/lfw.dat   
  inflating: /content/data/cfp_fp.dat.conf  
  inflating: /content/data/cfp_ff_list.npy  
  inflating: /content/data/agedb_30.dat.conf  
  inflating: /content/data/agedb_30.dat  
  inflating: /content/data/cfp_ff.dat  
  inflating: /content/data/cfp_ff.dat.conf  
  inflating: /content/data/cfp_fp_list.npy  


In [ ]:
!cp -r /content/drive/MyDrive/Ying/faces_extracted.zip /content
!unzip -q /content/faces_extracted.zip -d /content/

!cp -r /content/drive/MyDrive/Ying/IJBB.zip /content
!unzip -q /content/IJBB.zip -d /content
!cp -r /content/drive/MyDrive/IJB_meta.zip /content
!unzip -q /content/IJB_meta.zip -d /content
!cp -r /content/meta/IJBB_meta/* /content/IJBB/meta

In [ ]:
# !cp -r /content/drive/MyDrive/ijb-testsuite.tar /content
# !file ijb-testsuite.tar
# !tar -xf ijb-testsuite.tar
# !cp -r /content/drive/MyDrive/IJB_meta.zip /content
# !unzip -q /content/IJB_meta.zip -d /content
# !cp -r /content/meta/IJBB_meta/* /content/ijb/IJBB/meta
# !cp -r /content/meta/IJBC_meta/* /content/ijb/IJBC/meta

In [ ]:
import numpy as np
from tqdm import tqdm
import os
import json
from PIL import Image
import cv2

import torch
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader

from xiaoying.validation import evaluate_utils, evaluate
from xiaoying import net
from xiaoying.data import CustomImageFolderDataset, val_dataset
from xiaoying.head import AdaFace, SubAdaFace
from xiaoying.utils import EarlyStopping, load_checkpoint, load_weights, split_parameters

In [ ]:
root = '/content/faces_extracted'
print("Num claass: ", len(os.listdir(root)))

train_transforms = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

train_dataset = CustomImageFolderDataset('/content/faces_extracted',
                                         transform=train_transforms)
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    drop_last=True
)


val_ds = val_dataset(data_root='/content/data')
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False,
                        num_workers=4, pin_memory=True)

Num claass:  8631
loading validation data memfile
loading validation data memfile
loading validation data memfile
loading validation data memfile


In [ ]:
model_name = 'ir_50'
model = net.build_model(model_name)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

criterion = nn.CrossEntropyLoss()
head = AdaFace(embedding_size=512, classnum=8631, m=0.4, h=0.333, s=64., t_alpha=0.99)
head.to(device)

paras_wo_bn, paras_only_bn = split_parameters(model)

optimizer = torch.optim.SGD([{
            'params': paras_wo_bn + [head.kernel],
            'weight_decay': 1e-4
        }, {
            'params': paras_only_bn
        }], lr=0.01, momentum=0.9)

Total parameters: 43,585,600
Trainable parameters: 43,585,600
AdaFace with the following property
self.m 0.4
self.h 0.333
self.s 64.0
self.t_alpha 0.99


In [ ]:
def validate(model, dataname, val_loader, device):
    # Evaluate
    print('Evaluate step 1...')
    mean_acc = evaluate.evaluate1(model, val_loader, device)

    print('Evaluate step 2...')
    r = evaluate.evaluate2('', model, 'IR', 'IJBB', 256, device=device)

    return mean_acc, r

In [ ]:
def train(model, head, train_loader, optimizer,
          criterion, epochs, device, **kwargs):
    early_stopping = EarlyStopping('/content', 'checkpoint_backups/',
                               '/content/drive/MyDrive/checkpoints',
                                   model, head, optimizer)

    # load checkpoint
    start = 0 if not kwargs.get('filename') else \
                    load_checkpoint(kwargs.get('filename'), model, head, optimizer,
                                    early_stopping, device)
    optimizer = torch.optim.SGD([{
            'params': paras_wo_bn + [head.kernel],
            'weight_decay': 1e-4
        }, {
            'params': paras_only_bn
        }], lr=0.0001, momentum=0.9)

    for epoch in range(start, epochs):
        # ==================== Training ==================== #
        model.train()

        train_loss = []
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        print('\nTrain...')
        for images, labels in pbar:

            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # forward
            embedings, norm = model(images)
            cos_theta = head(embedings, norm, labels)
            loss = criterion(cos_theta, labels)

            # backward
            loss.backward()
            optimizer.step()

            train_loss.append(loss.item())

            # update tqdm
            pbar.set_postfix({
                "loss": f"{np.mean(train_loss):.4f}",
                "lr": optimizer.param_groups[0]["lr"]
            })

        early_stopping._save() # save backup

        # ==================== Validation ==================== #
        model.eval()
        mean_acc, r = validate(model, kwargs.get('dataname'), val_loader, device)

        early_stopping(**{
            'acc': r['0.0001'],
            'epoch': epoch+1,
            'fpr_1e-4': r['0.0001'],
            'fpr_1e-5': r['1e-05'],
        })
        print(f'Loss: {np.mean(train_loss):.4f} - Acc: {mean_acc:.4f}\n')

        if early_stopping.stop:
            print("Early Stopping")
            break

In [ ]:
args = {
    'root': '/content',
    'model_name': model_name,
    'filename': '/content/drive/MyDrive/checkpoints/checkpoint_5.pth',
    'dataname': 'IJBB'
}

train(model, head, train_loader, optimizer, criterion, epochs=7, device=device, **args)

Successfully load model statedict with epoch 5.



Epoch 6/7:   0%|          | 0/12257 [00:00<?, ?it/s]


Train...


Epoch 6/7: 100%|██████████| 12257/12257 [3:57:01<00:00,  1.16s/it, loss=3.2520, lr=0.0001]


Save temporary checkpoint to checkpoint_backups/temp_checkpoint.pth

Evaluate step 1...


...

100%|██████████| 890/890 [15:31<00:00,  1.05s/it]


Feature Shape: (227630 , 512) .
(3323,) (3323,)
(3377,) (3377,)
(6700,) (6700,)
total_templates (227630,) (227630,)
Finish Calculating 0 template features.
gallery_templates_feature (1845, 512)
gallery_unique_subject_ids (1845,)
(220926,) (220926,)
Finish Calculating 0 template features.
Finish Calculating 2000 template features.
Finish Calculating 4000 template features.
Finish Calculating 6000 template features.
Finish Calculating 8000 template features.
Finish Calculating 10000 template features.
probe_mixed_templates_feature (10270, 512)
probe_mixed_unique_subject_ids (10270,)
(10270, 512)
(1845, 512)
similarity shape (10270, 1845)
(10270, 1845)
top1 = 0.9344693281402142
top5 = 0.9628042843232717
top10 = 0.9712755598831548
18937880
(10270,)
neg_sims num = 18937880
after sorting , neg_sims num = 1027
far = 0.0100000000 pr = 0.7574488802 th = 0.5973101123
far = 0.1000000000 pr = 0.8994157741 th = 0.4425542206
Finish Calculating 0 template features.
Finish Calculating 2000 template fe

Epoch 7/7:   0%|          | 0/12257 [00:00<?, ?it/s]


Train...


Epoch 7/7:  21%|██        | 2517/12257 [48:32<3:07:49,  1.16s/it, loss=3.2275, lr=0.0001]